# MP30 — Image Patches and Window Attention

[Brief](../stage_07_transformers/MP30_image_patches_and_window_attention/README.md) · [Validation](../stage_07_transformers/MP30_image_patches_and_window_attention/VALIDATION.md). Prerequisite: MP29. Explore token ordering, exact reconstruction, connectivity and analytical storage without training or downloading data.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import matplotlib.pyplot as plt
root=Path.cwd() if (Path.cwd()/'src/rssr').exists() else Path.cwd().parent
sys.path.insert(0,str(root/'stage_07_transformers/MP30_image_patches_and_window_attention'))
from window_tools import patchify,unpatchify,connectivity,score_counts


## 1. Patch shapes and positional order

For H×W×C pixels and square patch sizeP, N=(H/P)(W/P) tokens each containP²C numbers. We require exact divisibility and use row-major patch order; within each patch, rows, columns then channels are flattened. No padding or compression occurs. A learned embedding would add a matrix fromP²C to feature widthD and might lose information; that is separate from patchification.

Hand example: a4×4 single-channel image with values0…15 and P=2 yields first token[0,1,4,5], second[2,3,6,7]. Shuffling token rows changes reconstructed spatial positions unless the inverse permutation is recorded.

In [ ]:
x=np.arange(16).reshape(4,4,1);tokens=patchify(x,2)
np.testing.assert_array_equal(tokens[0],[0,1,4,5]);np.testing.assert_array_equal(tokens[1],[2,3,6,7])
np.testing.assert_array_equal(unpatchify(tokens,x.shape,2),x)
print(tokens)
shuffled=unpatchify(tokens[[1,0,2,3]],x.shape,2)
assert not np.array_equal(shuffled,x)


## 2. Global, local and shifted connectivity

Consider a4×4 token grid and2×2 windows. Global attention permits every pair (256). Local windows permit64 pairs. Shift the window boundary by one cell: clipped edge windows produce36 pairs and allow communication across some old boundaries. No cyclic wrap is allowed; first and last edge tokens are not neighbors.

This explicit clipped-boundary diagram is educational, not an implementation of Swin's cyclic-shift and mask machinery. Two layers (local then shifted) provide paths between144 ordered pairs, still fewer than global256. Connectivity only says a path is possible, not that learned weights use it.

In [ ]:
local=connectivity(4,4,2);shifted=connectivity(4,4,2,1)
assert not local[1,2] and shifted[1,2] and not shifted[0,3]
reached=(shifted.astype(int)@local.astype(int))>0
print('pairs local / shifted / two layers:',local.sum(),shifted.sum(),reached.sum())
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,label,array in zip(axes,['Global','Local 2×2','Shifted, clipped','Two-layer paths'],[np.ones_like(local),local,shifted,reached]):
    ax.imshow(array,vmin=0,vmax=1,cmap='Blues');ax.set(title=label,xlabel='Input token',ylabel='Output token')
fig.tight_layout();plt.show()


## 3. Exact real-image round trips

The recorded experiment starts with a synthetic integer ramp, then a valid8×8 RGB crop from a checksum-pinned NAIP training scene. Patch sizes1,2,4 preserve exact dtype and values. No normalization or resampling is needed; stored uint16 values remain unchanged. The lesson reads recorded evidence; the separate experiment script checks the source hash before re-running.

In [ ]:
record=json.loads((root/'reports/mp30-windows.json').read_text())
assert record['source']['split']=='train'
assert len(record['rows'])==6 and all(r['exact_roundtrip'] for r in record['rows'])
print(record['source'])
for row in record['rows']:print(row)


## 4. Complexity and storage

Global attention scores need N² entries and O(N²d) pairwise work. For complete M×M token windows, each token sees M² keys: NM² entries and O(NM²d) pairwise work. Projections and feed-forward work are excluded. Fixed M makes this term linear in N.

At4×4 tokens, M=2: global256 entries/1024 float32 bytes; local64/256 bytes. These are analytical score counts, not peak RAM. Clipped shifted windows may have fewer entries. Do not allocate large dense adjacency matrices just to estimate memory.

In [ ]:
assert score_counts(4,4,2)['global_float32_bytes']==1024
assert score_counts(4,4,2)['local_float32_bytes']==256
fig,ax=plt.subplots(figsize=(7,4))
for field,label in [('global_float32_bytes','Global'),('local_float32_bytes','Local 2×2')]:
    ax.loglog([r['tokens'] for r in record['counts']],[r[field]/2**20 for r in record['counts']],'o-',label=label)
ax.set(xlabel='Token count N',ylabel='Analytical score storage (MiB, float32)',title='One head; excludes projections and gradients')
ax.legend();ax.grid(alpha=.2);fig.tight_layout();plt.show()


## Practice and solutions

Predict token count for a32×32 image with P=4; explain why reversing patch order changes reconstruction; decide whether two shifted layers equal global attention. Answers:64 tokens; token order encodes location implicitly; no, the measured two-layer adjacency is144 rather than256 pairs.

Reflection: reversible tokenization and attention connectivity solve different problems. Explicit boundary conventions avoid accidental cross-edge communication. The next lesson compares an existing compact window-based SR baseline with a CNN.

Limitations: no learned embedding, softmax weights or SR accuracy; adjacency examples use clipped boundaries, not faithful Swin masks. No frozen evaluation or new acquisition. NAIP is public-domain aerial data, USDA/OpenSR attribution retained, dataset-card MIT; no reflectance claim. Learner mastery remains unassessed.

References: [Swin Transformer](https://arxiv.org/abs/2103.14030) for shifted windows; [SwinIR](https://arxiv.org/abs/2108.10257) for restoration. This lesson illustrates connectivity rather than reproducing either network.

In [ ]:
assert patchify(np.zeros((32,32,3)),4).shape==(64,48)
try:patchify(np.zeros((7,8,3)),2)
except ValueError as error:print('Expected divisibility rejection:',error)
else:raise AssertionError('Implicit padding was not permitted')
